In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("sales_dates.csv")

In [4]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"])
df.dtypes

Order_ID               int64
Order_Date    datetime64[us]
Product                  str
Category                 str
Region                   str
Sales                  int64
Quantity               int64
dtype: object

In [5]:
df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Day"] = df["Order_Date"].dt.day
df["Day_Name"] = df["Order_Date"].dt.day_name()
df["Quarter"] = df["Order_Date"].dt.quarter
df[
    [
        "Order_Date",
        "Year",
        "Month",
        "Month_Name",
        "Day",
        "Day_Name",
        "Quarter"
    ]
]

,Order_Date,Year,Month,Month_Name,Day,Day_Name,Quarter
0,2026-01-05,2026,1,January,5,Monday,1
1,2026-01-08,2026,1,January,8,Thursday,1
2,2026-01-12,2026,1,January,12,Monday,1
3,2026-01-18,2026,1,January,18,Sunday,1
4,2026-02-03,2026,2,February,3,Tuesday,1
5,2026-02-09,2026,2,February,9,Monday,1
6,2026-02-15,2026,2,February,15,Sunday,1
7,2026-02-22,2026,2,February,22,Sunday,1
8,2026-03-04,2026,3,March,4,Wednesday,1
9,2026-03-11,2026,3,March,11,Wednesday,1


In [6]:
earliest_Order = df["Order_Date"].min()
latest_Order = df["Order_Date"].max()

# Use .days instead of .dt.days for scalar Timedelta objects
number_between = (latest_Order - earliest_Order).days

print("Earliest Order:", earliest_Order)
print("Latest Order:", latest_Order)
print("Days Between:", number_between)

Earliest Order: 2026-01-05 00:00:00
Latest Order: 2026-06-17 00:00:00
Days Between: 163


In [7]:
order_filter = df[
    df["Order_Date"].between(
        "2026-02-01",
        "2026-03-31"
    )
]
order_filter

,Order_ID,Order_Date,Product,Category,Region,Sales,Quantity,Year,Month,Month_Name,Day,Day_Name,Quarter
4,1005,2026-02-03,Laptop,Electronics,North,80000,2,2026,2,February,3,Tuesday,1
5,1006,2026-02-09,Headphones,Accessories,West,5000,4,2026,2,February,9,Monday,1
6,1007,2026-02-15,Keyboard,Accessories,South,3500,3,2026,2,February,15,Sunday,1
7,1008,2026-02-22,Monitor,Electronics,West,30000,2,2026,2,February,22,Sunday,1
8,1009,2026-03-04,Mouse,Accessories,North,1800,6,2026,3,March,4,Wednesday,1
9,1010,2026-03-11,Laptop,Electronics,South,70000,2,2026,3,March,11,Wednesday,1
10,1011,2026-03-18,Headphones,Accessories,North,5500,5,2026,3,March,18,Wednesday,1
11,1012,2026-03-25,Monitor,Electronics,South,28000,2,2026,3,March,25,Wednesday,1


In [8]:
monthly_report = df.groupby("Month").agg(
    Total_Sales = ("Sales", "sum"),
    Average_Sales = ("Sales", "mean"),
    Total_Quantity = ("Quantity", "count"),
    Number_of_Orders = ("Order_ID", "count")
)
monthly_report

,Total_Sales,Average_Sales,Total_Quantity,Number_of_Orders
Month,,,,
1,104500,26125.000000,4,4
2,118500,29625.000000,4,4
3,105300,26325.000000,4,4
4,76800,25600.000000,3,3
5,110200,36733.333333,3,3
6,72400,36200.000000,2,2


In [9]:
quarter_report = df.groupby("Quarter").agg(
    Total_Sales = ("Sales", "sum"),
    Average_Sales = ("Sales", "mean"),
)
quarter_report

,Total_Sales,Average_Sales
Quarter,,
1,328300,27358.333333
2,259400,32425.000000


In [10]:
weekday_sales = df.groupby("Day_Name")["Sales"].sum()

highestweekday = weekday_sales.idxmax()
highestamount = weekday_sales.max()

print(f"The weekday with the highest total sales is {highestweekday} with {highestamount:.2f}")

The weekday with the highest total sales is Tuesday with 193400.00


In [11]:
df["Month_Period"] = df["Order_Date"].dt.to_period("M")  # e.g., '2026-01'
df["Quarter"] = df["Order_Date"].dt.to_period("Q")       # e.g., '2026Q1'
df["Weekday"] = df["Order_Date"].dt.day_name()           # e.g., 'Monday'

In [13]:
monthly_report = (
    df.groupby("Month_Period")
    .agg(
        Total_Sales=("Sales", "sum"),
        Average_Sales=("Sales", "mean"),
        Total_Quantity=("Quantity", "sum"),
        # Assuming Order_ID tracks individual orders; use nunique for unique orders
        Number_of_Orders=("Order_ID", "nunique"), 
    )
    .reset_index()
)
monthly_report

,Month_Period,Total_Sales,Average_Sales,Total_Quantity,Number_of_Orders
0,2026-01,104500,26125.000000,12,4
1,2026-02,118500,29625.000000,11,4
2,2026-03,105300,26325.000000,15,4
3,2026-04,76800,25600.000000,9,3
4,2026-05,110200,36733.333333,8,3
5,2026-06,72400,36200.000000,5,2


In [14]:
monthly_report["Sales_Percentage"] = (
    monthly_report["Total_Sales"] / monthly_report["Total_Sales"].sum() * 100
).round(2)

In [15]:
# Q1: Highest Sales Month
highest_month = monthly_report.loc[monthly_report["Total_Sales"].idxmax(), "Month_Period"]
highest_sales = monthly_report["Total_Sales"].max()

In [16]:
# Q2: Lowest Sales Month
lowest_month = monthly_report.loc[monthly_report["Total_Sales"].idxmin(), "Month_Period"]
lowest_sales = monthly_report["Total_Sales"].min()

In [17]:
# Q3: Highest Revenue Quarter
quarterly_sales = df.groupby("Quarter")["Sales"].sum()
best_quarter = quarterly_sales.idxmax()
best_quarter_sales = quarterly_sales.max()

In [18]:
# Q4: Highest Revenue Weekday
weekday_sales = df.groupby("Weekday")["Sales"].sum()
best_weekday = weekday_sales.idxmax()
best_weekday_sales = weekday_sales.max()

In [20]:
print("=== MONTHLY SALES INTELLIGENCE REPORT ===")
print(monthly_report.to_string(index=False))
print("\n" + "="*40)

print(f"1. Highest Sales Month: {highest_month} ({highest_sales:,.2f})")
print(f"2. Lowest Sales Month:  {lowest_month} ({lowest_sales:,.2f})")
print(f"3. Top Quarter:         {best_quarter} ({best_quarter_sales:,.2f})")
print(f"4. Top Weekday:         {best_weekday} ({best_weekday_sales:,.2f})")

=== MONTHLY SALES INTELLIGENCE REPORT ===
Month_Period  Total_Sales  Average_Sales  Total_Quantity  Number_of_Orders  Sales_Percentage
     2026-01       104500   26125.000000              12                 4             17.78
     2026-02       118500   29625.000000              11                 4             20.16
     2026-03       105300   26325.000000              15                 4             17.92
     2026-04        76800   25600.000000               9                 3             13.07
     2026-05       110200   36733.333333               8                 3             18.75
     2026-06        72400   36200.000000               5                 2             12.32

1. Highest Sales Month: 2026-02 (118,500.00)
2. Lowest Sales Month:  2026-06 (72,400.00)
3. Top Quarter:         2026Q1 (328,300.00)
4. Top Weekday:         Tuesday (193,400.00)


In [21]:
monthly_report.to_csv("monthly_sales_report.csv", index=False)

In [22]:
quarterly_report = (
    df.groupby("Quarter")
    .agg(
        Total_Sales=("Sales", "sum"),
        Average_Sales=("Sales", "mean"),
        Total_Quantity=("Quantity", "sum"),
        Number_of_Orders=("Order_ID", "nunique"),
    )
    .reset_index()
)
quarterly_report.to_csv("quarterly_sales_report.csv", index=False)